# Séance 2 — Mémoire, RAG agentique et premier graphe
**Agentic AI** · Master 2 I2A · Université Paris 8<br>
**Enseignant :** Pr Mehdi Ammi — Laboratoire LIASD (EA 4383)<br>
**Version :** octobre 2026

*Note sur l'usage de l'IA : le contenu pédagogique (objectifs, progression, exercices, solutions) a été conçu par l'auteur. Claude Code (Anthropic) a été utilisé pour la mise en forme du notebook.*

---

> **🎯 Objectifs de la séance**
> - Comprendre les embeddings et la recherche vectorielle
> - Construire un RAG agentique : la recherche devient un outil que l'agent choisit d'utiliser
> - Donner une mémoire à l'agent (checkpointer, `Store`)
> - Sortir de `create_agent` pour écrire son propre `StateGraph`, avec un routeur

> **⏱️ Durée totale : 6 h**
> - Partie 1 — Notions : ~1 h
> - Partie 2 — TP guidé puis autonome : ~4 h
> - Livrable de fin de séance : ~50 min

> **📥 Livrable de fin de séance :** à déposer sur **Moodle**, dans l'espace « Livrable 2 » du cours, **à la fin de la journée**. Le sujet se trouve dans la section *Livrable 2*, en fin de notebook.

---

## Prérequis techniques

```bash
# Depuis le dossier seance2
pip install -r requirements.txt
cp .env.example .env          # puis choisir l'option A (Ollama) ou B (API personnelle)
python common/check_setup.py  # doit afficher « OK »

# Option A — LLM local avec Ollama (https://ollama.com)
ollama serve
ollama pull qwen3:8b
```

Le modèle d'embedding (`paraphrase-multilingual-MiniLM-L12-v2`) est téléchargé au premier lancement (~500 Mo) et tourne en local, quelle que soit votre option LLM.

**Données utilisées :** une trentaine de résumés d'articles arXiv sur votre thème de veille (téléchargés par le notebook) ; pour le livrable, les PDF fournis dans `data/livrable2/`.

**Projet fil rouge :** un assistant de veille scientifique qui cherche des articles, les lit, les résume et rédige une synthèse critique. Il s'enrichit à chaque séance.

> **Organisation des travaux pratiques**
> - Les cellules **NE PAS MODIFIER — Démonstration** sont fournies : exécutez-les et lisez le code.
> - Les cellules **✏️ À COMPLÉTER** contiennent des `TODO`.
> - L'accompagnement diminue au fil de la séance. Chaque exercice précise sa **modalité** : compléter le code fourni, implémenter à partir des étapes indiquées, réutiliser et étendre votre propre code, puis concevoir la solution en autonomie à partir des critères de réussite.

> **Indices :** 2 niveaux disponibles auprès de l'enseignant.

## Installation des bibliothèques de la séance
Si votre environnement a été créé pour la séance 1, il lui manque des bibliothèques de la séance 2 (`chromadb`, `arxiv`, `matplotlib`…). Exécutez la cellule ci-dessous **une seule fois**, puis redémarrez le noyau (*Kernel → Restart Kernel*) avant de continuer.

> `%pip` s'exécute **dans le notebook** : il installe les paquets dans l'environnement du noyau utilisé. Dans un terminal, utilisez `python -m pip install …` après avoir activé votre environnement.

### Choisir le moteur d'embeddings
Les embeddings sont calculés par `common/embeddings.py`. Deux moteurs, réglés dans `.env` :

| `EMBEDDING_BACKEND` | Modèle | Quand l'utiliser |
| --- | --- | --- |
| `sentence-transformers` (par défaut) | `paraphrase-multilingual-MiniLM-L12-v2`, local, via PyTorch | cas général |
| `ollama` | `paraphrase-multilingual`, servi par Ollama, sans PyTorch | **Mac à processeur Intel**, ou si l'import de `sentence_transformers` échoue |

Pour le moteur Ollama : `ollama pull paraphrase-multilingual` dans un terminal, ajoutez la ligne `EMBEDDING_BACKEND=ollama` à votre `.env`, puis redémarrez le noyau.

In [ ]:
%pip install -q langgraph langchain langchain-openai python-dotenv arxiv pypdf requests chromadb sentence-transformers matplotlib rank-bm25

## Questions de rappel
Sans note, sans ordinateur.
1. Quelles sont les 4 composantes d'un agent ?
2. Que lit le modèle pour décider d'appeler un outil ?
3. Que se passe-t-il sans `recursion_limit` ?

*Vos réponses :*

---
# Partie 1 — Embeddings et recherche documentaire
---

### Rappels de cours
- La **fenêtre de contexte** est limitée et coûteuse : on ne peut pas y mettre tout un corpus.
- Un **embedding** transforme un texte en vecteur ; deux textes proches en sens ont des vecteurs proches (**similarité cosinus**).
- Une **base vectorielle** (ici ChromaDB) stocke ces vecteurs et retrouve les plus proches d'une requête.
- On **découpe** les documents en morceaux (taille, recouvrement) et on garde des **métadonnées** (titre, année) pour citer ses sources.
- **RAG classique :** on cherche toujours avant de répondre. **RAG agentique :** la recherche est un outil, l'agent décide quand chercher et reformule sa requête.

## Exercice 1 — Similarité sémantique

> **Modalité :** exécuter le code fourni et analyser les résultats.

Exécutez la cellule, puis expliquez **2 résultats surprenants** de la matrice de similarité.

In [ ]:
# NE PAS MODIFIER — Démonstration
# Le dossier common/ se trouve dans le dossier de la séance
from common.llm import get_llm
llm = get_llm()

import numpy as np
import matplotlib.pyplot as plt
from common.embeddings import encoder, BACKEND  # moteur choisi dans .env

print("Moteur d'embeddings :", BACKEND)

PHRASES = [
    "Les agents LLM utilisent des outils externes.",
    "Un modèle de langage peut appeler des API.",
    "Le chat dort sur le canapé.",
    "Le félin se repose sur le sofa.",
    "La banque a augmenté ses taux.",
    "Nous nous sommes assis sur le banc au bord de la rivière.",
    "L'apprentissage par renforcement optimise une récompense.",
    "Reinforcement learning maximizes a reward signal.",
    "Il pleut à Paris.",
    "Il ne pleut pas à Paris.",
]

vecteurs = encoder(PHRASES)  # un vecteur normalisé par phrase
similarite = vecteurs @ vecteurs.T

plt.figure(figsize=(7, 6))
plt.imshow(similarite, vmin=0, vmax=1, cmap="viridis")
plt.colorbar(label="similarité cosinus")
plt.xticks(range(len(PHRASES)), range(len(PHRASES)))
plt.yticks(range(len(PHRASES)), [p[:35] for p in PHRASES])
plt.tight_layout()
plt.show()

*Deux résultats surprenants et leur explication :*

1. …
2. …

## Exercice 2 — Indexation et recherche vectorielle

> **Modalité :** compléter le code fourni, puis implémenter à partir des étapes indiquées.

Les trois fonctions sont de moins en moins détaillées : `chunk` est à compléter, `index_corpus` indique les étapes, `search_corpus` ne donne que la signature. L'exercice est terminé lorsque **tous les tests passent**.

In [ ]:
# NE PAS MODIFIER — Démonstration
import json
import arxiv
from pathlib import Path

CORPUS_PATH = Path("data/corpus/articles.json")


def download_corpus(query: str = "LLM agents", n: int = 30) -> list[dict]:
    """Télécharge n résumés arXiv."""
    recherche = arxiv.Search(query=query, max_results=n)
    docs = [
        {"id": r.get_short_id(), "titre": r.title, "annee": r.published.year,
         "texte": r.summary.replace("\n", " ")}
        for r in arxiv.Client().results(recherche)
    ]
    CORPUS_PATH.write_text(json.dumps(docs, ensure_ascii=False, indent=1))
    return docs


docs = json.loads(CORPUS_PATH.read_text()) if CORPUS_PATH.exists() else download_corpus()
print(len(docs), "documents")

In [ ]:
# ✏️ À COMPLÉTER
import chromadb
from common.embeddings import chroma_embedding_function

client = chromadb.PersistentClient(path="chroma_db")
# même moteur d'embedding qu'à l'exercice 1
ef = chroma_embedding_function()


# a) Compléter le code
def chunk(texte: str, taille: int = 500, recouvrement: int = 100) -> list[str]:
    """Découpe un texte en morceaux de `taille` caractères qui se recouvrent de `recouvrement`."""
    if not texte:
        return []
    pas = ...  # TODO : de combien de caractères avance-t-on entre deux morceaux ?
    return [texte[i:i + taille] for i in range(0, ..., pas)]  # TODO : jusqu'où va i ?


# b) Implémenter à partir des étapes indiquées
def index_corpus(docs: list[dict], nom: str = "veille") -> chromadb.Collection:
    """Découpe chaque document et stocke ses morceaux dans une collection Chroma recréée à chaque appel.

    Chaque morceau garde en métadonnées le titre, l'année et l'id du document.
    """
    # 1. supprimer la collection si elle existe (client.delete_collection), puis la créer
    #    avec client.create_collection(nom, embedding_function=ef)
    # 2. pour chaque document, découper doc["texte"] avec chunk()
    # 3. préparer trois listes : ids (uniques), documents (les morceaux), metadatas (titre, annee, id)
    # 4. collection.add(ids=..., documents=..., metadatas=...) puis renvoyer la collection
    ...


# c) Implémenter à partir de la signature. Indice : collection.query(query_texts=[...], n_results=k)
#     renvoie des listes "documents", "metadatas" et "distances".
def search_corpus(collection: chromadb.Collection, query: str, k: int = 4) -> list[dict]:
    """Renvoie les k morceaux les plus proches : [{"texte", "titre", "annee", "score"}]."""
    ...

In [ ]:
# NE PAS MODIFIER — Démonstration
# Tests
assert chunk("") == []
assert chunk("a" * 400) == ["a" * 400]
morceaux = chunk("x" * 1200, taille=500, recouvrement=100)
assert len(morceaux) == 3 and all(len(m) <= 500 for m in morceaux)
assert morceaux[0][-100:] == morceaux[1][:100], "les morceaux doivent se recouvrir"

collection = index_corpus(docs)
assert collection.count() >= len(docs)

resultats = search_corpus(collection, docs[0]["texte"][:200], k=3)
assert len(resultats) == 3
assert {"texte", "titre", "annee", "score"} <= resultats[0].keys()
assert resultats[0]["titre"] == docs[0]["titre"], "un extrait du document doit retrouver ce document en premier"
print("Tous les tests passent.")

## Exercice 3 — La recherche documentaire comme outil

> **Modalité :** compléter le code fourni.

Complétez le code pour transformer votre `search_corpus` en outil, puis posez les questions. **Repérez dans les traces quand l'agent décide de chercher, ou non**, et avec quelle requête.

In [ ]:
# ✏️ À COMPLÉTER
from langchain.agents import create_agent
from langchain_core.tools import tool


@tool
def chercher_corpus(requete: str) -> str:
    """TODO : quand l'agent doit-il utiliser cet outil, et comment formuler la requête ?"""
    res = ...  # TODO : appeler votre search_corpus de l'exercice 2
    return "\n\n".join(f"[{r['titre']} ({r['annee']})] {r['texte']}" for r in res)


agent_rag = create_agent(model=llm, tools=[chercher_corpus],
                         system_prompt="Tu réponds en citant les titres des articles utilisés.")

for q in ["Bonjour !",
          "Quels articles du corpus parlent d'évaluation des agents ?",
          "Compare deux approches de planification décrites dans le corpus."]:
    print("=" * 60, "\n", q)
    for m in agent_rag.invoke({"messages": [("user", q)]})["messages"]:
        m.pretty_print()

*Observations : quand l'agent cherche-t-il ? Ses requêtes sont-elles bonnes ? Modifiez votre docstring et observez la différence.*

## Exercice 4 — Mémoire de conversation

> **Modalité :** compléter le code fourni.

Un **checkpointer** sauvegarde l'état de chaque conversation, identifiée par un `thread_id`. Complétez la cellule et vérifiez que deux `thread_id` donnent deux conversations indépendantes.

In [ ]:
# ✏️ À COMPLÉTER
from langgraph.checkpoint.memory import InMemorySaver

agent_memoire = create_agent(model=llm, tools=[chercher_corpus], checkpointer=InMemorySaver())

conv_a = {"configurable": {"thread_id": "A"}}
conv_b = {"configurable": {"thread_id": "B"}}

agent_memoire.invoke({"messages": [("user", "Je m'appelle Inès et je m'intéresse à la planification.")]}, conv_a)
# TODO : demander « Comment je m'appelle ? » dans la conversation A, puis dans la conversation B

---
# Partie 2 — Écrire son propre graphe
---

### Rappels de cours
- **Workflow** : le code fixe le chemin. **Agent** : le LLM le choisit. Commencer par le plus simple qui marche.
- Un `StateGraph` = un **état typé** partagé + des **nœuds** (fonctions) + des **arêtes** (fixes ou conditionnelles). Les cycles sont permis.
- `add_messages` fusionne les nouveaux messages dans l'état au lieu de l'écraser.
- **Routage** : un nœud classe la demande, une arête conditionnelle envoie vers la bonne branche.

## Exercice 5 — Construction d'un graphe LangGraph

> **Modalité :** compléter le code fourni.

Remplacez `create_agent` par votre propre graphe. Le premier nœud est fourni ; complétez les arêtes.

```
START → modele ──(appel d'outil ?)──oui──→ outils ──→ modele
                    └──non──→ END
```

In [ ]:
# ✏️ À COMPLÉTER
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition


class Etat(TypedDict):
    messages: Annotated[list, add_messages]


OUTILS_RAG = [chercher_corpus]
llm_outils = llm.bind_tools(OUTILS_RAG)


def modele(etat: Etat) -> dict:
    return {"messages": [llm_outils.invoke(etat["messages"])]}


g = StateGraph(Etat)
g.add_node("modele", modele)
g.add_node("outils", ToolNode(...))        # TODO : quels outils ?
g.add_edge(START, ...)                      # TODO
g.add_conditional_edges("modele", ..., {"tools": "outils", END: END})  # TODO : quelle condition ?
g.add_edge(..., ...)                        # TODO : le retour vers le modèle
graphe = g.compile(checkpointer=InMemorySaver())

print(graphe.get_graph().draw_mermaid())

---
# Travail en autonomie

Les exercices suivants s'appuient sur le code que vous avez déjà écrit.

## Exercice 6 — Routage des demandes

> **Modalité :** implémenter à partir des étapes indiquées.

Ajoutez un nœud `routeur` en tête du graphe de l'exercice 5. Les étapes sont données :
1. Définissez un modèle Pydantic `Route` avec un champ `destination` qui vaut `"corpus"` ou `"direct"`.
2. Écrivez le nœud `routeur` : il appelle `llm.with_structured_output(Route)` sur la dernière question et range la destination dans l'état.
3. Ajoutez une arête conditionnelle : `"corpus"` → votre nœud `modele` (avec outils), `"direct"` → un nouveau nœud qui répond sans outil.
4. Affichez le graphe en Mermaid et testez 3 questions.

In [ ]:
# ✏️ À COMPLÉTER
# Exercice 6 — routeur

## Exercice 7 — Comparaison de stratégies de découpage

> **Modalité :** réutiliser et étendre votre propre code.

Réutilisez vos fonctions `chunk`, `index_corpus` et `search_corpus` de l'exercice 2. Comparez deux stratégies de découpage (tailles, recouvrements, ou découpage par phrases) sur 5 questions dont vous connaissez la réponse. **Choisissez vous-même la métrique** et justifiez votre choix.

In [ ]:
# ✏️ À COMPLÉTER
# Exercice 7 — mini-étude

## Exercice 8 — Mémoire courte et longue

> **Modalité :** réutiliser et étendre votre propre code.

Partez de votre graphe de l'exercice 6.
1. Quand l'historique dépasse N messages, résumez les anciens messages pour garder le contexte court.
2. Avec un `Store` LangGraph, retenez d'une conversation à l'autre les **thèmes suivis** et les **articles déjà lus**.

Documentation : [Memory dans LangGraph](https://docs.langchain.com/oss/python/langgraph/add-memory).

In [ ]:
# ✏️ À COMPLÉTER
# Exercice 8 — mémoire

## Exercice 9 — Projet fil rouge

> **Modalité :** concevoir la solution en autonomie, à partir des critères de réussite.

Reprenez votre agent de veille de la séance 1 et votre graphe de l'exercice 8.
1. **Sur papier** : dessinez le graphe de l'assistant, avec un routeur vers « recherche de nouveaux articles » (arXiv) ou « question sur le corpus ». Justifiez chaque choix workflow / agent. Ajoutez une photo du schéma au dépôt.
2. Implémentez-le : l'assistant ajoute automatiquement au corpus les articles trouvés et répond en citant ses sources.

**Critères :** le schéma correspond au code, les réponses citent leurs sources, un article déjà indexé n'est pas ajouté deux fois.

In [ ]:
# ✏️ À COMPLÉTER
# Exercice 9 — projet

---
# Livrable 2 — Assistant documentaire

> **📥 Dépôt sur Moodle — à la fin de la journée**<br>
> Déposez ce notebook exécuté (`S2_memoire_rag_graphe.ipynb`) et le schéma du graphe dans l'espace **« Livrable 2 »** du cours sur Moodle.


**Sujet.** Sur le corpus de PDF fourni dans `data/livrable2/` :
1. Indexez le corpus et exposez la recherche comme outil.
2. Écrivez un `StateGraph` avec un **routeur** : question sur le corpus, question générale, ou question hors sujet (refus poli).
3. Ajoutez un checkpointer pour que l'agent se souvienne de la conversation.
4. Exportez le schéma du graphe (Mermaid) et répondez aux 6 questions ci-dessous, **dans la même conversation**.

**Critères de réussite :** le routeur oriente correctement au moins 5 questions sur 6, les réponses sur le corpus citent leur source, la question de suivi est bien comprise, le schéma correspond au code.

In [ ]:
# ✏️ À COMPLÉTER
# [À compléter par l'enseignant selon le corpus fourni]
QUESTIONS_LIVRABLE = [
    "Question sur le corpus n° 1",
    "Question sur le corpus n° 2",
    "Question sur le corpus n° 3",
    "Question de suivi qui fait référence à la réponse précédente",
    "Question générale (sans rapport avec le corpus)",
    "Question hors sujet à refuser",
]

# Votre graphe ici

---
# Exercices d'approfondissement

1. **Recherche hybride :** combinez BM25 (`rank-bm25`) et recherche vectorielle, puis comparez sur les questions de la mini-étude.
2. **Reranking :** réordonnez les résultats avec un modèle cross-encoder avant de les passer au LLM.
3. **Rappel@k :** mesurez la qualité de votre recherche sur un petit jeu de questions annotées.
4. **Mémoire sémantique :** extrayez automatiquement des faits de la conversation et stockez-les dans le `Store`.

In [ ]:
# ✏️ À COMPLÉTER
# Exercices d'approfondissement

---
## Journal de bord

- Un échec observé aujourd'hui :
- Sa cause :
- Ce que j'ai fait pour le corriger :